![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/text-summarization/Summarization_Annotator.ipynb)

# Document Summarization with the `Summarization` Annotator

`Summarization` is a high level, **task oriented** annotator: you say *what* you want (how long, what style, what to focus on) and it decides *how* to produce it, selecting and downloading a model, building prompts, setting safe generation defaults, and handling documents that are longer than the model context.

It supports three methods, each with an automatically selected default model:

| Method | What it does | Default model | Best for |
|--------|--------------|---------------|----------|
| `llm` (default) | Instruction tuned LLM (llama.cpp) generates a summary | `qwen3_4b_q8_0_gguf` | Highest quality, controllable via style/focus; GPU recommended |
| `encoder_decoder` | Purpose built abstractive model (DistilBART) | `distilbart_xsum_12_6` | Fast fluent abstractive summaries |
| `extractive` | Selects the most central original sentences (PacSum style centrality + MMR) | `all_mpnet_base_v2` | CPU only, high throughput, source faithful (no hallucination) |

Every method takes `DOCUMENT` in and returns `DOCUMENT` out, with rich metadata describing what happened. This notebook walks through all of it end to end.

## 1. Setup

Install and start Spark NLP. On Google Colab, use the setup script.

In [ ]:
# Only run this cell if you are on Google Colab
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Summarization, SummarizationModel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.sql import functions as F

# For the LLM method a GPU is recommended. On a CPU-only machine keep gpu=False and set
# .setGpuLayers(0) on the llm summarizer (shown later).
spark = sparknlp.start()

print('Spark NLP version:', sparknlp.version())
print('Apache Spark version:', spark.version)

## 2. Quickstart

A minimal pipeline is a `DocumentAssembler` feeding a `Summarization` stage. The `method` is chosen at build time; the default (or an explicit) model is resolved and downloaded when you call `fit()`.

We start with the **extractive** method because it is CPU friendly and downloads only a small sentence embeddings model.

In [ ]:
document_assembler = DocumentAssembler() \
    .setInputCol('text') \
    .setOutputCol('document')

summarizer = Summarization() \
    .setInputCols(['document']) \
    .setOutputCol('summary') \
    .setMethod('extractive') \
    .setMaxSummaryLength(60)

pipeline = Pipeline(stages=[document_assembler, summarizer])

In [ ]:
article = (
    'Renewable energy capacity grew at a record pace last year, driven mainly by solar and wind. '
    'Global installations rose by nearly 50 percent compared with the previous year, according to '
    'the International Energy Agency. China accounted for the largest share of new capacity, followed '
    'by the European Union and the United States. Falling equipment costs and supportive government '
    'policies were the main drivers of the expansion. Analysts caution, however, that grid '
    'infrastructure and permitting delays could slow deployment in the coming years. The agency '
    'expects renewables to overtake coal as the largest source of electricity generation before the '
    'end of the decade if current trends continue.'
)

data = spark.createDataFrame([[article]]).toDF('text')
model = pipeline.fit(data)   # resolves + downloads the default extractive model
result = model.transform(data)

result.select('summary.result').show(truncate=False)

### The output is a `DOCUMENT` with transparency metadata

Every summary annotation carries metadata describing exactly what the annotator did: which method and model ran, the inference engine, estimated token counts, how many chunks were used, and (for extractive) how many sentences were selected.

In [ ]:
result.select(F.explode('summary').alias('s')) \
    .select(F.col('s.result').alias('summary'), F.col('s.metadata').alias('metadata')) \
    .show(truncate=False)

## 3. Comparing the three methods on the same document

The same task level API works across methods; only `setMethod` (and the model it resolves) changes. Below we run **extractive** and **encoder_decoder** on the same article and compare.

> The **llm** method downloads a ~4 GB GGUF model and prefers a GPU. It is shown in section 6 with CPU safe settings.

In [ ]:
def summarize(method, text, **params):
    s = Summarization().setInputCols(['document']).setOutputCol('summary').setMethod(method)
    for k, v in params.items():
        getattr(s, 'set' + k[0].upper() + k[1:])(v)
    p = Pipeline(stages=[document_assembler, s])
    df = spark.createDataFrame([[text]]).toDF('text')
    out = p.fit(df).transform(df)
    row = out.select('summary.result', 'summary.metadata').first()
    return row[0][0], row[1][0]

ext_summary, ext_meta = summarize('extractive', article, maxSummaryLength=60)
enc_summary, enc_meta = summarize('encoder_decoder', article, maxSummaryLength=60)

print('EXTRACTIVE  (method=%s, model=%s)' % (ext_meta['method'], ext_meta['model']))
print(ext_summary)
print()
print('ENCODER_DECODER  (method=%s, model=%s)' % (enc_meta['method'], enc_meta['model']))
print(enc_summary)

Notice the difference: **extractive** returns sentences copied verbatim from the source (safe, no hallucination), while **encoder_decoder** rephrases the content into new, fluent sentences (abstractive).

## 4. Shaping the summary: length, style, and focus

Task level parameters control the result without touching model internals:

* `setMaxSummaryLength` / `setMinSummaryLength`: approximate word budget.
* `setSummaryStyle`: `concise`, `detailed`, or `bullets` (llm prompt).
* `setFocus`: a free text hint, e.g. *"financial impact"* (llm prompt).

For the encoder_decoder method, length is enforced through the model's decoding; for the llm method it is written into the prompt. Below we contrast a short vs. a longer abstractive summary.

In [ ]:
short_summary, _ = summarize('encoder_decoder', article, maxSummaryLength=20)
long_summary,  _ = summarize('encoder_decoder', article, maxSummaryLength=120)

print('SHORT (<=20 words):')
print(short_summary)
print()
print('LONGER (<=120 words):')
print(long_summary)

## 5. Long documents are handled automatically

When a document is longer than the model's context window, `Summarization` splits it at sentence boundaries into overlapping chunks, summarizes each chunk, and then combines and re summarizes the intermediate summaries (hierarchical map/reduce). You never split documents yourself.

`setLongDocumentStrategy` accepts:
* `auto` (default): summarize directly when it fits, otherwise fall back to hierarchical.
* `hierarchical`: always chunk, summarize, and reduce.
* `truncate`: cut to the context limit and summarize once.

`setChunkSize` (approximate tokens) and `setChunkOverlap` (sentences shared between chunks) give finer control. The `numChunks` metadata field tells you how many chunks were used.

In [ ]:
# Build a long document by concatenating several paragraphs.
long_document = ' '.join([article] * 8)   # ~2,600 words

long_summarizer = Summarization() \
    .setInputCols(['document']) \
    .setOutputCol('summary') \
    .setMethod('encoder_decoder') \
    .setMaxSummaryLength(90) \
    .setLongDocumentStrategy('hierarchical') \
    .setChunkSize(400) \
    .setChunkOverlap(1)

long_pipeline = Pipeline(stages=[document_assembler, long_summarizer])
long_df = spark.createDataFrame([[long_document]]).toDF('text')
long_out = long_pipeline.fit(long_df).transform(long_df)

row = long_out.select('summary.result', 'summary.metadata').first()
print('Number of chunks used:', row[1][0]['numChunks'])
print('Strategy:', row[1][0]['longDocumentStrategy'])
print()
print(row[0][0])

## 6. The LLM method (instruction tuned, most controllable)

The `llm` method runs an instruction tuned GGUF model through llama.cpp. The annotator owns the prompt, the system prompt, and reasoning suppression, so you only provide task settings. Style and focus are honored through the prompt.

**On CPU:** call `setGpuLayers(0)`. **On GPU:** use the Spark NLP GPU package and leave `gpuLayers` at its default (99). The default model is ~4 GB and will download on first `fit()`.

In [ ]:
llm_summarizer = Summarization() \
    .setInputCols(['document']) \
    .setOutputCol('summary') \
    .setMethod('llm') \
    .setMaxSummaryLength(70) \
    .setSummaryStyle('bullets') \
    .setFocus('the main drivers and the main risk') \
    .setGpuLayers(0)          # set to 99 (default) on a GPU cluster

llm_pipeline = Pipeline(stages=[document_assembler, llm_summarizer])
llm_out = llm_pipeline.fit(data).transform(data)

row = llm_out.select('summary.result', 'summary.metadata').first()
print('engine:', row[1][0]['engine'], '| model:', row[1][0]['model'])
print()
print(row[0][0])

## 7. Real world batch: summarizing many documents at once

Because `Summarization` is a normal Spark NLP stage, a fitted model summarizes an entire DataFrame in one distributed pass, one summary per input row. This is the typical production pattern: a table of articles, tickets, filings, or abstracts in, a table of summaries out.

In [ ]:
documents = [
    ('energy', article),
    ('health', (
        'A large clinical study reported that a new once-daily pill reduced the risk of stroke in '
        'high-risk patients by about a third compared with the standard treatment. The trial '
        'followed more than fifteen thousand participants across twelve countries for four years. '
        'Side effects were generally mild, though a small number of patients discontinued the drug '
        'because of dizziness. Regulators are expected to review the results later this year, and '
        'the manufacturer said it would seek approval in several major markets.')),
    ('tech', (
        'The company unveiled its next-generation processor, which it says delivers a forty percent '
        'improvement in performance per watt over the previous generation. The chip is built on a '
        'newer manufacturing process and adds dedicated hardware for on-device machine learning. '
        'Analysts said the efficiency gains could help the company win back customers in the laptop '
        'market, where battery life has become a key selling point. Shipments are scheduled to begin '
        'next quarter.')),
]

batch_df = spark.createDataFrame(documents, ['category', 'text'])

batch_summarizer = Summarization() \
    .setInputCols(['document']) \
    .setOutputCol('summary') \
    .setMethod('extractive') \
    .setMaxSummaryLength(40)

batch_model = Pipeline(stages=[document_assembler, batch_summarizer]).fit(batch_df)
batch_result = batch_model.transform(batch_df)

batch_result.select(
    'category',
    F.col('summary.result').getItem(0).alias('summary'),
    F.col('summary.metadata').getItem(0)['sentencesSelected'].alias('sentences_selected'),
).show(truncate=100)

## 8. Save and reload a fitted pipeline

Saving a fitted `Summarization` pipeline persists the underlying model weights, so it reloads and runs **offline**, no re download and no network access required.

In [ ]:
path = '/tmp/summarization_pipeline'
batch_model.write().overwrite().save(path)

reloaded = PipelineModel.load(path)
reloaded.transform(batch_df).select(
    'category', F.col('summary.result').getItem(0).alias('summary')
).show(truncate=100)

## 9. Choosing a method

| If you need... | Use |
|----------------|-----|
| No hallucination, exact source sentences, CPU only, high throughput | `extractive` |
| Fluent rephrased summaries, fast, no GPU required | `encoder_decoder` |
| The highest quality and control via style/focus, GPU available | `llm` |

All three share the same API, output type, and metadata, so you can prototype with `extractive`, then switch a single `setMethod` call to `encoder_decoder` or `llm` when you are ready.

### Notes
* Parameters that do not apply to the chosen method are ignored with a logged warning (e.g. `setNumBeams` on `llm`, `setMmrLambda` on `encoder_decoder`).
* `setMinSummaryLength` must be `<= setMaxSummaryLength` (validated at `fit()`).
* This annotator runs at the DataFrame level and is **not** supported in `LightPipeline`.
* For a custom model, pass `setModel('<name-from-models-hub>')` matching the selected method's engine.